In [2]:
import duckdb
import ollama
import re

# Setup
DB_PATH = "../data/omop_clinical.duckdb"
MODEL_NAME = "qwen2.5-coder:7b"

def get_unmapped_conditions(limit=5):
    """Fetch a sample of unique unmapped clinical conditions from DuckDB."""
    print("🔌 Connecting to DuckDB to fetch unmapped conditions...")
    with duckdb.connect(DB_PATH) as con:
        query = """
            SELECT DISTINCT condition_source_value
            FROM condition_occurrence
            WHERE condition_concept_id = 0
            AND condition_source_value IS NOT NULL
            LIMIT ?
        """
        return [row[0] for row in con.execute(query, [limit]).fetchall()]

def ai_semantic_normalization(raw_term):
    """Uses local LLM to normalize a messy clinical term into a core standard name."""
    system_prompt = """
    You are an expert Clinical Data Informatician.
    Your task is to normalize raw, messy clinical text into a clean, core medical term.
    RULES:
    1. Respond ONLY with the core medical term.
    2. Do NOT include any numeric IDs.
    3. Do NOT include any tags in parentheses like '(disorder)', '(finding)', or '(person)'.
    4. Keep it as short and precise as possible.
    """
    
    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': f"Raw clinical text: '{raw_term}'"}
            ]
        )
        
        # Clean the output: remove quotes and any remaining parentheses blocks
        clean_text = response['message']['content'].strip().strip("'").strip('"')
        clean_text = re.sub(r'\([^)]*\)', '', clean_text).strip()
        
        return clean_text
    except Exception as e:
        return f"Error: {e}"

# EXECUTION BLOCK
print("⚙️ STARTING AI-ASSISTED SEMANTIC MAPPING (V2: FUZZY SEARCH)\n" + "-"*50)

unmapped_terms = get_unmapped_conditions(limit=5)

if not unmapped_terms:
    print("✅ No unmapped conditions found! Everything is perfect.")
else:
    print(f"⚠️ Found unmapped conditions. Sending a sample of {len(unmapped_terms)} to Qwen...\n")
    
    for term in unmapped_terms:
        print(f"🔸 Raw Term (Failed): {term}")
        
        # 1. Ask AI to normalize the text
        normalized_term = ai_semantic_normalization(term)
        print(f"✨ AI Suggestion:     {normalized_term}")
        
        # 2. Search DuckDB using Fuzzy Matching across ALL domains
        with duckdb.connect(DB_PATH) as con:
            search_query = """
                SELECT concept_id, concept_name, domain_id 
                FROM concept 
                WHERE vocabulary_id = 'SNOMED' 
                AND LOWER(concept_name) LIKE LOWER(?)
                LIMIT 1
            """
            # Using % wildcards to find the term anywhere in the concept name
            match = con.execute(search_query, [f"%{normalized_term}%"]).fetchone()
            
            if match:
                print(f"🎯 DB Match Found!   ID: {match[0]} | Name: '{match[1]}' | Domain: {match[2]}")
            else:
                print("❌ Still no match. Term might be too vague or missing in standard vocab.")
        print("-" * 50)

⚙️ STARTING AI-ASSISTED SEMANTIC MAPPING (V2: FUZZY SEARCH)
--------------------------------------------------
🔌 Connecting to DuckDB to fetch unmapped conditions...
⚠️ Found unmapped conditions. Sending a sample of 5 to Qwen...

🔸 Raw Term (Failed): Medication review due (situation)
✨ AI Suggestion:     medication review
🎯 DB Match Found!   ID: 35609168 | Name: 'Medication review by community nurse' | Domain: Procedure
--------------------------------------------------
🔸 Raw Term (Failed): Limited social contact (finding)
✨ AI Suggestion:     social isolation
🎯 DB Match Found!   ID: 4232870 | Name: 'At risk for social isolation' | Domain: Observation
--------------------------------------------------
🔸 Raw Term (Failed): Serving in military service (finding)
✨ AI Suggestion:     Military Service
🎯 DB Match Found!   ID: 1074831 | Name: 'Military service member of United States Coast Guard' | Domain: Observation
--------------------------------------------------
🔸 Raw Term (Failed): Spu